# Benchmark local pelo Ollama

O notebook usa as mesmas funções do script: uma chamada por questão, associação
por `id_prova` e `Nquestao` com `validate="one_to_one"`, e salvamento incremental.
O aquecimento é separado das questões e das métricas. Execute a partir da raiz
do repositório ou de `Lmm_teste_1`.

As saídas antigas foram removidas. Executar a célula de benchmark abaixo chama o
Ollama local e pode baixar o modelo se ele estiver ausente.


In [ ]:
from pathlib import Path
import sys

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "Lmm_teste_1" / "teste_modelo_aut.py").is_file())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from Lmm_teste_1.teste_modelo_aut import (
    carregar_entrada, rodar_modelo, carregar_progresso,
    configuracao, associar_resultados, imprimir_relatorio, caminho_saida,
)


In [ ]:
arquivo = "provas_tjdft_com_gabarito"
modelo = "gemma4:e4b-it-q4_K_M"
pasta_saida = RAIZ / "resultados" / "notebook"
retomar = False  # Para continuar a mesma execução, mude explicitamente para True.
caminho, prova_2015 = carregar_entrada(arquivo, RAIZ / "database_csv")
prova_2015.head()


In [ ]:
rodar_modelo(
    modelos=(modelo,), arquivos=(arquivo,), retomar=retomar,
    pasta_csv=RAIZ / "database_csv", pasta_saida=pasta_saida,
)


A célula abaixo também pode ser usada após uma interrupção, sem novas inferências.
Cada resposta salva contém `modelo`, `arquivo`, `id_prova` e `Nquestao`.
Somente questões processadas com gabarito A–E entram na acurácia; anuladas e
sem gabarito são preservadas e excluídas da avaliação. Respostas inválidas e
falhas de inferência em questões avaliáveis contam como não acertos.
O relatório identifica execuções parciais. A contagem de respostas sem letra
inclui falhas de inferência, portanto essas duas contagens podem se sobrepor.
`carregamento` descreve a alocação em CPU/GPU, não sua utilização computacional.
`possivel_truncamento` é apenas uma heurística e fica desconhecida sem contagem
de tokens. O JSON de progresso é a fonte da retomada; não o edite manualmente.


In [ ]:
progresso = caminho_saida(modelo, arquivo, pasta_saida).with_suffix(".progresso.json")
resultados = carregar_progresso(
    progresso, configuracao(modelo, arquivo, caminho, prova_2015), prova_2015,
)
df_final = associar_resultados(prova_2015, resultados)
imprimir_relatorio(df_final, modelo, arquivo)
df_final.head()
